# Mi primer notebook de Jupyter

Una plantilla para estudiar y reutilizar. Usa **Python 3** como kernel y ejecuta las celdas de arriba abajo. No necesita datos externos.

## 1. Cómo usarlo

- **Shift + Enter**: ejecuta la celda y avanza.
- **Ctrl + Enter**: ejecuta sin avanzar.
- **Esc**, luego **A** o **B**: inserta una celda arriba o abajo en modo comando.
- **Esc**, luego **M** o **Y**: cambia la celda a Markdown o código.
- **Tab**: autocompleta; **Shift + Tab**: muestra ayuda de una función.
- Usa el botón de guardar para conservar cambios y resultados.

Los atajos pueden variar según el entorno. Las celdas comparten memoria: una variable existe después de ejecutar la celda que la define. El número `In [n]` indica el orden de ejecución, no la posición de la celda.

**Buena práctica:** antes de compartir, usa la opción de reiniciar el kernel y ejecutar todas las celdas. Así compruebas que el notebook funciona sin variables antiguas.

## 2. Texto con Markdown

Haz doble clic en esta celda para ver cómo está escrita; ejecútala para volver a renderizarla.

**Negrita**, *cursiva*, `código en línea` y [documentación de Jupyter](https://docs.jupyter.org/).

> Usa texto para explicar qué preguntas quieres responder y cómo interpretas los resultados.

| Tipo de celda | Para qué sirve |
|---|---|
| Markdown | Explicaciones, enlaces, tablas y fórmulas |
| Código | Instrucciones que ejecuta el kernel |

También puedes escribir fórmulas con LaTeX: $x^2$ o una ecuación centrada:

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

Una imagen se inserta con `![Descripción](ruta/imagen.png)`. Esa ruta debe existir cuando compartas el notebook.

## 3. Preparación del entorno

La siguiente celda instala las bibliotecas en el entorno del kernel actual. Requiere acceso a Internet y modifica ese entorno; puedes omitirla si ya tienes todo instalado.

`%pip` es un **comando mágico de IPython**, no sintaxis normal de Python. En un proyecto real conviene registrar las versiones de las dependencias.

In [ ]:
%pip install pandas matplotlib ipywidgets

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Markdown, display

%matplotlib inline
print('Entorno listo')

## 4. Python y memoria compartida

`print()` produce texto. La última expresión de una celda se muestra automáticamente. Para mostrar varios resultados enriquecidos, utiliza `display()`.

Cambia `nombre` y vuelve a ejecutar esta celda y la siguiente.

In [ ]:
nombre = 'Estudiante'
precios = [10, 15, 20]

def aplicar_descuento(precio, porcentaje=10):
    """Calcula el precio final con un descuento entre 0 y 100."""
    if not 0 <= porcentaje <= 100:
        raise ValueError('El porcentaje debe estar entre 0 y 100')
    return precio * (1 - porcentaje / 100)

print(f'Hola, {nombre}')
[aplicar_descuento(precio) for precio in precios]

In [ ]:
display(Markdown(f'**{nombre}**, este texto se ha generado desde Python.'))
print('La variable precios viene de la celda anterior:', precios)

## 5. Tablas y análisis con pandas

Creamos datos de ejemplo en memoria. `DataFrame` representa una tabla; Jupyter la muestra con formato HTML. pandas y matplotlib son bibliotecas adicionales, no funciones propias de Jupyter.

In [ ]:
df = pd.DataFrame({
    'producto': ['Libro', 'Cuaderno', 'Lapiz', 'Mochila'],
    'precio': [18.0, 5.0, 1.5, 30.0],
    'unidades': [4, 10, 25, 3],
})
df['ingresos'] = df['precio'] * df['unidades']

display(df)
display(df.describe())
print(f"Ingresos totales: {df['ingresos'].sum():.2f}")

## 6. Visualización

Modifica los datos o el título y vuelve a ejecutar. `plt.show()` muestra el gráfico dentro del notebook.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df['producto'], df['ingresos'], color='steelblue')
ax.set(title='Ingresos por producto', xlabel='Producto', ylabel='Ingresos')
fig.tight_layout()
plt.show()

## 7. Ayuda y comandos mágicos

`?` consulta documentación; `%timeit` mide varias ejecuciones de una expresión; `%%time` mide una celda completa. Los resultados de tiempo dependen de tu equipo.

Otros comandos útiles: `%pwd` muestra el directorio de trabajo y `%lsmagic` enumera los comandos disponibles. También existe `!comando` para ejecutar comandos del sistema: úsalo solo con instrucciones que entiendas y consideres seguras.

In [ ]:
aplicar_descuento?

In [ ]:
%timeit -n 100 -r 3 df['ingresos'].sum()

In [ ]:
%%time
total = sum(range(100_000))
print(total)

## 8. Interactividad con widgets

Mueve el control para recalcular una tabla sin editar el código. Necesita un kernel activo y un cliente compatible con ipywidgets; una vista estática no permite interactuar.

Si solo aparece una representación de texto del control, revisa la instalación de ipywidgets en tu entorno de Jupyter.

In [ ]:
def mostrar_descuento(porcentaje):
    tabla = df[['producto', 'precio']].copy()
    tabla['precio_final'] = tabla['precio'].map(
        lambda precio: aplicar_descuento(precio, porcentaje)
    )
    display(tabla.round(2))

control = widgets.interact(
    mostrar_descuento,
    porcentaje=widgets.IntSlider(value=10, min=0, max=50, step=5, description='Descuento %'),
)

## 9. Leer y escribir archivos

Este ejemplo guarda un CSV y vuelve a leerlo en una carpeta temporal que se limpia al terminar, sin sobrescribir tus archivos.

Para conservarlo en tu proyecto, usa por ejemplo `df.to_csv('ventas.csv', index=False)` en una celda nueva. Las rutas relativas parten del directorio de trabajo del kernel; puedes consultarlo con `%pwd`.

In [ ]:
with TemporaryDirectory(prefix='ejemplo_jupyter_') as carpeta:
    ruta = Path(carpeta) / 'ventas.csv'
    df.to_csv(ruta, index=False)
    df_leido = pd.read_csv(ruta)
    display(df_leido)

pd.testing.assert_frame_equal(df, df_leido)
print('Los datos guardados y leidos coinciden')

## 10. Errores y comprobaciones

Si ejecutas `aplicar_descuento(100, 150)` en una celda nueva, verás un `ValueError` y su traceback. Lee la última línea para identificar el error y las anteriores para localizarlo.

La siguiente celda incluye comprobaciones sencillas. Una aserción fallida detiene esa celda y muestra el problema.

In [ ]:
assert aplicar_descuento(100, 0) == 100
assert aplicar_descuento(100, 20) == 80
assert aplicar_descuento(100, 100) == 0
assert df['ingresos'].sum() == 249.5
print('Comprobaciones correctas')

## 11. Tu plantilla de trabajo

Duplica este notebook y sustituye los ejemplos con esta estructura:

1. **Objetivo:** qué quieres averiguar.
2. **Preparación:** bibliotecas, configuración y origen de los datos.
3. **Exploración:** primeras filas, tipos, datos ausentes y estadísticas.
4. **Análisis:** cálculos y gráficos.
5. **Conclusiones:** interpretación, limitaciones y próximos pasos.

### Ejercicios

- Añade un producto y comprueba cómo cambia el gráfico; actualiza también la aserción del total.
- Filtra los productos con ingresos superiores a 60.
- Crea un gráfico de unidades vendidas.
- Explica el resultado en una celda Markdown.

### Compartir y exportar

Guarda el `.ipynb` para conservar código, texto y salidas. Según tu cliente, usa el menú de exportación o `jupyter nbconvert --to html plantilla_jupyter.ipynb` en una terminal para generar HTML; este comando requiere nbconvert. El HTML es estático y no sustituye al kernel para ejecutar código o widgets.

**Antes de compartir:** reinicia el kernel, ejecuta todo, guarda y revisa que las salidas no contengan datos privados, rutas sensibles ni credenciales.